# BosonSR Binary Transition: Basic Analysis

This notebook runs a compact smoke-analysis of the non-CBC `BosonSR_binary` waveform and its independent numerical-cloud-mass flavour, `BosonSR_binary_num`. The GW wrapper loads the binary-transition implementation from `../henry/julia_wf/functions.jl`.


In [ ]:
using Pkg

candidate_dirs = unique([
    abspath(@__DIR__),
    abspath(pwd()),
    joinpath(abspath(@__DIR__), "GW_free_params.jl"),
    joinpath(abspath(pwd()), "GW_free_params.jl"),
])

project_dir_idx = findfirst(dir -> isfile(joinpath(dir, "Project.toml")) && isfile(joinpath(dir, "src", "GW.jl")), candidate_dirs)
project_dir_idx === nothing && error("Could not find GW_free_params.jl project directory from pwd=$(pwd()) and @__DIR__=$(@__DIR__)")
project_dir = candidate_dirs[project_dir_idx]
repo_dir = dirname(project_dir)

wrapper_source = joinpath(project_dir, "src", "waveforms", "bosonSR_binary_transition.jl")
binary_source = joinpath(repo_dir, "henry", "julia_wf", "functions.jl")
@assert isfile(wrapper_source) wrapper_source
@assert isfile(binary_source) binary_source

Pkg.activate(project_dir)
include(joinpath(project_dir, "src", "GW.jl"))
using .GW
using .GW: BosonSR_binary, BosonSR_binary_num, Pol, PolAbs, Strain, SNR, FisherMatrix, CE1Id_coordinates, CE1Id, ETLS, ETLMR, _available_waveforms
using Plots

default(size=(850, 420), linewidth=2, grid=true)

println("GW project: ", project_dir)
println("GW wrapper: ", wrapper_source)
println("binary-transition implementation: ", binary_source)


## Difference Diagnostics

The analytical and numerical Henry models share the same frequency, phase, envelope, and angular factors. Their direct waveform differences are therefore expected to mostly trace the difference between analytical `q_c(alpha, m_i)` and the numerical cloud-mass ratio `Mcloud / (M - Mcloud)`. The helpers below report maximum absolute and relative differences while ignoring entries where both models are effectively zero.


In [ ]:
function max_abs_rel_difference(numerical, analytical; atol=0.0)
    num = collect(skipmissing(vec(numerical)))
    ana = collect(skipmissing(vec(analytical)))
    @assert length(num) == length(ana)

    finite_idx = findall(k -> isfinite(num[k]) && isfinite(ana[k]), eachindex(num))
    isempty(finite_idx) && return (max_abs=NaN, max_rel=NaN, abs_index=nothing, rel_index=nothing)

    abs_diff = abs.(num[finite_idx] .- ana[finite_idx])
    denom = max.(abs.(ana[finite_idx]), atol)
    valid_rel_idx = findall(>(0.0), denom)
    rel_diff = fill(NaN, length(finite_idx))
    rel_diff[valid_rel_idx] = abs_diff[valid_rel_idx] ./ denom[valid_rel_idx]

    abs_pos = argmax(abs_diff)
    rel_pos = argmax(replace(rel_diff, NaN => -Inf))
    return (
        max_abs=abs_diff[abs_pos],
        max_rel=rel_diff[rel_pos],
        abs_index=finite_idx[abs_pos],
        rel_index=finite_idx[rel_pos],
    )
end

function print_difference_summary(label, numerical, analytical; atol=0.0)
    summary = max_abs_rel_difference(numerical, analytical; atol=atol)
    println(label)
    println("  max absolute difference: ", summary.max_abs, " at flat index ", summary.abs_index)
    println("  max relative difference: ", summary.max_rel, " at flat index ", summary.rel_index)
    return summary
end


## Waveform and Parameters

`dL` is in Gpc. The example uses a nearby source (`1e-6` Gpc = 1 kpc) and a black-hole mass that places the transition near 100 Hz. `Gamma_abs` is supplied in the natural-energy units expected by Henry's waveform.

In [ ]:
model = GW.BosonSR_binary()
model_num = GW.BosonSR_binary_num()

q = 1.0e-3
M_solar = 1.0e-3
alpha_sr = 0.21
Gamma_abs = 1.0e-14
intrinsic = (q, M_solar, alpha_sr, Gamma_abs)

dL = 1.0e-6
theta = 1.0
phi = 2.0
iota = 0.4
psi = 0.7
tcoal = 0.2
phiCoal = 0.0

f_center = GW.waveform._henry_central_frequency_Hz(model, q, M_solar, alpha_sr)
f_center_num = GW.waveform._henry_central_frequency_Hz(model_num, q, M_solar, alpha_sr)
fmin = max(2.0, 0.7 * f_center)
fmax = 1.3 * f_center
f = collect(range(fmin, fmax, length=800))

println("available analytical model: ", GW._available_waveforms("BosonSR_binary"))
println("available numerical model: ", GW._available_waveforms("BosonSR_binary_num"))
println("analytical parameter names: ", GW.waveform._parameter_names(model))
println("numerical parameter names: ", GW.waveform._parameter_names(model_num))
println("central frequency [Hz]: ", f_center)
println("numerical central frequency [Hz]: ", f_center_num)
println("analysis band [Hz]: ", (first(f), last(f)))
println("Henry function module: ", GW.waveform.HenryBinaryTransition)


## Source-Frame Polarizations

In [ ]:
h_plus, h_cross = Pol(model, f, intrinsic, dL, iota)

plot(f, abs.(h_plus), label="analytical |h+|", xlabel="f [Hz]", ylabel="strain / Hz", yscale=:log10)
plot!(f, abs.(h_cross), label="analytical |hx|")
vline!([f_center], label="f_center", linestyle=:dash, color=:black)


## Numerical Cloud-Mass Source-Frame Polarizations

`BosonSR_binary_num` uses the same waveform interface as `BosonSR_binary`, but forces Henry's numerical cloud-mass calculation for the amplitude normalization.


In [ ]:
h_plus_num, h_cross_num = Pol(model_num, f, intrinsic, dL, iota)

plot(f, abs.(h_plus_num), label="numerical |h+|", xlabel="f [Hz]", ylabel="strain / Hz", yscale=:log10)
plot!(f, abs.(h_cross_num), label="numerical |hx|")
vline!([f_center_num], label="f_center", linestyle=:dash, color=:black)


In [ ]:
M_eV = M_solar * GW.waveform.HenryBinaryTransition.HenryWF.Msol_in_eV
boson_mass = alpha_sr / (GW.waveform.HenryBinaryTransition.HenryWF.G * M_eV)
cloud_mass_num = GW.waveform.HenryBinaryTransition.HenryWF.compute_cloud_mass_numerical(alpha_sr, M_eV, boson_mass; spin=0.99)
qc_analytical = GW.waveform.HenryBinaryTransition.HenryWF.q_c(alpha_sr, model.m_i)
qc_numerical = cloud_mass_num / (M_eV - cloud_mass_num)

println("analytical q_c: ", qc_analytical)
println("numerical Mcloud / (M - Mcloud): ", qc_numerical)
println("absolute q_c difference: ", abs(qc_numerical - qc_analytical))
println("relative q_c difference: ", abs(qc_numerical - qc_analytical) / abs(qc_analytical))


In [ ]:
plot(f, abs.(h_plus), label="analytical |h+|", xlabel="f [Hz]", ylabel="strain / Hz", yscale=:log10)
plot!(f, abs.(h_plus_num), label="numerical |h+|", linestyle=:dash)
plot!(f, abs.(h_cross), label="analytical |hx|")
plot!(f, abs.(h_cross_num), label="numerical |hx|", linestyle=:dash)
vline!([f_center], label="f_center", linestyle=:dot, color=:black)


In [ ]:
polarization_difference_summary = (
    plus = print_difference_summary("source-frame h+ numerical vs analytical", abs.(h_plus_num), abs.(h_plus)),
    cross = print_difference_summary("source-frame hx numerical vs analytical", abs.(h_cross_num), abs.(h_cross)),
)


## Detector Strain

In [ ]:
strain_ce = Strain(
    model,
    CE1Id_coordinates,
    f,
    intrinsic,
    dL,
    theta,
    phi,
    iota,
    psi,
    tcoal,
    phiCoal,
)

plot(f, abs.(strain_ce), label="CE1Id |h_det|", xlabel="f [Hz]", ylabel="detector strain / Hz", yscale=:log10)
vline!([f_center], label="f_center", linestyle=:dash, color=:black)

## Numerical Cloud-Mass Detector Strain


In [ ]:
strain_ce_num = Strain(
    model_num,
    CE1Id_coordinates,
    f,
    intrinsic,
    dL,
    theta,
    phi,
    iota,
    psi,
    tcoal,
    phiCoal,
)

plot(f, abs.(strain_ce), label="analytical CE1Id |h_det|", xlabel="f [Hz]", ylabel="detector strain / Hz", yscale=:log10)
plot!(f, abs.(strain_ce_num), label="numerical CE1Id |h_det|", linestyle=:dash)
vline!([f_center], label="f_center", linestyle=:dot, color=:black)


In [ ]:
strain_difference_summary = print_difference_summary(
    "CE1Id detector strain numerical vs analytical",
    abs.(strain_ce_num),
    abs.(strain_ce),
)


## SNR

In [ ]:
network = [ETLS, ETLMR, CE1Id]

M_solar_range = 10.0 .^ range(-4, -2, length=10)
alpha_range = collect(range(0.15, 0.30, length=10))
snr_matrix = fill(NaN, length(M_solar_range), length(alpha_range));

In [ ]:
for (i, M_solar_i) in enumerate(M_solar_range)
    for (j, alpha_j) in enumerate(alpha_range)
        intrinsic_ij = (q, M_solar_i, alpha_j, Gamma_abs)
        f_center_ij = GW.waveform._henry_central_frequency_Hz(model, q, M_solar_i, alpha_j)
        isfinite(f_center_ij) && f_center_ij > 0 || continue

        fmin_ij = max(2.0, 0.7 * f_center_ij)
        fmax_ij = 1.3 * f_center_ij
        fmax_ij > fmin_ij || continue

        try
            snr_matrix[i, j] = SNR(
                model, network, intrinsic_ij, dL, theta, phi, iota, psi, tcoal;
                fmin=fmin_ij, fmax=fmax_ij, res=80,
            )
        catch err
            @warn "SNR calculation failed" M_solar_i alpha_j exception=(err, catch_backtrace())
        end
    end
end

snr_matrix

In [ ]:
heatmap(
    M_solar_range, alpha_range, log10.(snr_matrix)';
    ylabel="alpha", xlabel="M_solar [M_sun]",
    title="SNR for BosonSR_binary model", colorbar_title="log10 SNR",
    xscale=:log10,
)

## Numerical Cloud-Mass SNR


In [ ]:
snr_matrix_num = fill(NaN, length(M_solar_range), length(alpha_range))

for (i, M_solar_i) in enumerate(M_solar_range)
    for (j, alpha_j) in enumerate(alpha_range)
        intrinsic_ij = (q, M_solar_i, alpha_j, Gamma_abs)
        f_center_ij = GW.waveform._henry_central_frequency_Hz(model_num, q, M_solar_i, alpha_j)
        isfinite(f_center_ij) && f_center_ij > 0 || continue

        fmin_ij = max(2.0, 0.7 * f_center_ij)
        fmax_ij = 1.3 * f_center_ij
        fmax_ij > fmin_ij || continue

        try
            snr_matrix_num[i, j] = SNR(
                model_num, network, intrinsic_ij, dL, theta, phi, iota, psi, tcoal;
                fmin=fmin_ij, fmax=fmax_ij, res=80,
            )
        catch err
            @warn "Numerical SNR calculation failed" M_solar_i alpha_j exception=(err, catch_backtrace())
        end
    end
end

snr_matrix_num


In [ ]:
heatmap(
    M_solar_range, alpha_range, log10.(snr_matrix_num)';
    ylabel="alpha", xlabel="M_solar [M_sun]",
    title="SNR for BosonSR_binary_num model", colorbar_title="log10 SNR",
    xscale=:log10,
)


In [ ]:
heatmap(
    M_solar_range, alpha_range, log10.(snr_matrix_num ./ snr_matrix)';
    ylabel="alpha", xlabel="M_solar [M_sun]",
    title="Numerical / analytical SNR ratio", colorbar_title="log10 ratio",
    xscale=:log10,
)


In [ ]:
snr_difference_summary = print_difference_summary(
    "network SNR grid numerical vs analytical",
    snr_matrix_num,
    snr_matrix,
)


In [ ]:
qc_analytical_grid = [
    GW.waveform.HenryBinaryTransition.HenryWF.q_c(alpha_j, model.m_i)
    for M_solar_i in M_solar_range, alpha_j in alpha_range
]

qc_numerical_grid = [
    begin
        M_eV_ij = M_solar_i * GW.waveform.HenryBinaryTransition.HenryWF.Msol_in_eV
        boson_mass_ij = alpha_j / (GW.waveform.HenryBinaryTransition.HenryWF.G * M_eV_ij)
        cloud_mass_ij = GW.waveform.HenryBinaryTransition.HenryWF.compute_cloud_mass_numerical(alpha_j, M_eV_ij, boson_mass_ij; spin=0.99)
        cloud_mass_ij / (M_eV_ij - cloud_mass_ij)
    end
    for M_solar_i in M_solar_range, alpha_j in alpha_range
]

qc_difference_summary = print_difference_summary(
    "q_c normalization numerical vs analytical on intrinsic grid",
    qc_numerical_grid,
    qc_analytical_grid,
)


## Small Fisher Matrix

This is intentionally low resolution. Increase `res` after the smoke run is stable.

In [ ]:
fisher = FisherMatrix(
    model, CE1Id, intrinsic, dL, theta, phi, iota, psi, tcoal, phiCoal;
    res=200, fmin=fmin, fmax=fmax, rho_thres=nothing,
)

display(fisher)
heatmap(log10.(abs.(fisher)), xlabel="parameter index", ylabel="parameter index", title="|Fisher|", colorbar_title="log10 abs")

## Small Fisher Matrix: Numerical Cloud-Mass Model

This uses the same low-resolution smoke settings as the analytical Fisher matrix.


In [ ]:
fisher_num = FisherMatrix(
    model_num, CE1Id, intrinsic, dL, theta, phi, iota, psi, tcoal, phiCoal;
    res=200, fmin=fmin, fmax=fmax, rho_thres=nothing,
)

display(fisher_num)
heatmap(log10.(abs.(fisher_num)), xlabel="parameter index", ylabel="parameter index", title="|Fisher| for BosonSR_binary_num", colorbar_title="log10 abs")


In [ ]:
fisher_difference_summary = print_difference_summary(
    "CE1Id Fisher matrix numerical vs analytical",
    abs.(fisher_num),
    abs.(fisher),
)


## Median SNR over Extrinsic-Parameter Priors

For each point on the intrinsic grid, this section evaluates the network SNR for 20 common Monte Carlo draws from the catalog priors: `cos(theta), cos(iota) ~ Uniform(-1, 1)`, `phi, phiCoal ~ Uniform(0, 2pi)`, `psi ~ Uniform(0, pi)`, and `tcoal ~ Uniform(0, 1)` day. The plotted value is the median over those draws. `phiCoal` is sampled as part of the full extrinsic tuple but is absent from the SNR API because a constant phase does not change the SNR.

In [ ]:
using Random, Statistics

n_extrinsic_samples = 20
prior_rng = MersenneTwister(20260715)
extrinsic_samples = [
    (
        theta=acos(2.0 * rand(prior_rng) - 1.0),
        phi=2.0 * pi * rand(prior_rng),
        iota=acos(2.0 * rand(prior_rng) - 1.0),
        psi=pi * rand(prior_rng),
        tcoal=rand(prior_rng),
        phiCoal=2.0 * pi * rand(prior_rng),
    )
    for _ in 1:n_extrinsic_samples
]

function network_snr_from_polabs(model, network, intrinsic, dL, sample, pol_abs; fmin, fmax, res)
    detector_snrs = [
        SNR(
            model, detector, intrinsic, dL, sample.theta, sample.phi, sample.iota, sample.psi, sample.tcoal;
            fmin=fmin, fmax=fmax, res=res, ampl_precomputation=pol_abs,
        )
        for detector in network
    ]
    return sqrt(sum(abs2, detector_snrs))
end

In [ ]:
prior_res = 80

function median_network_snr_grid(model_for_grid; prior_res=80)
    median_matrix = fill(NaN, length(M_solar_range), length(alpha_range))

    for (i, M_solar_i) in enumerate(M_solar_range)
        for (j, alpha_j) in enumerate(alpha_range)
            intrinsic_ij = (q, M_solar_i, alpha_j, Gamma_abs)
            f_center_ij = GW.waveform._henry_central_frequency_Hz(model_for_grid, q, M_solar_i, alpha_j)
            isfinite(f_center_ij) && f_center_ij > 0 || continue

            fmin_ij = max(2.0, 0.7 * f_center_ij)
            fmax_ij = 1.3 * f_center_ij
            fcut_ij = min(GW.waveform._fcut(model_for_grid, intrinsic_ij...), fmax_ij)
            fcut_ij > fmin_ij || continue
            fgrid_ij = 10.0 .^ range(log10(fmin_ij), log10(fcut_ij), length=prior_res)

            sampled_snrs = Float64[]
            for sample in extrinsic_samples
                try
                    pol_abs = PolAbs(model_for_grid, fgrid_ij, intrinsic_ij, dL, sample.iota)
                    push!(sampled_snrs, network_snr_from_polabs(
                        model_for_grid, network, intrinsic_ij, dL, sample, pol_abs;
                        fmin=fmin_ij, fmax=fmax_ij, res=prior_res,
                    ))
                catch err
                    @warn "Sampled SNR calculation failed" model=typeof(model_for_grid) M_solar_i alpha_j exception=(err, catch_backtrace())
                end
            end

            finite_snrs = filter(isfinite, sampled_snrs)
            isempty(finite_snrs) || (median_matrix[i, j] = median(finite_snrs))
        end
    end

    return median_matrix
end

median_snr_matrix = median_network_snr_grid(model; prior_res=prior_res)
median_snr_matrix


In [ ]:
heatmap(
    M_solar_range, alpha_range, log10.(median_snr_matrix)';
    ylabel="alpha", xlabel="M_solar [M_sun]",
    title="Analytical median network SNR over $(n_extrinsic_samples) extrinsic-prior draws",
    colorbar_title="log10 median SNR", xscale=:log10,
)


## Numerical Cloud-Mass Median SNR over Extrinsic-Parameter Priors


In [ ]:
median_snr_matrix_num = median_network_snr_grid(model_num; prior_res=prior_res)
median_snr_matrix_num


In [ ]:
heatmap(
    M_solar_range, alpha_range, log10.(median_snr_matrix_num)';
    ylabel="alpha", xlabel="M_solar [M_sun]",
    title="Numerical median network SNR over $(n_extrinsic_samples) extrinsic-prior draws",
    colorbar_title="log10 median SNR", xscale=:log10,
)


In [ ]:
heatmap(
    M_solar_range, alpha_range, log10.(median_snr_matrix_num ./ median_snr_matrix)';
    ylabel="alpha", xlabel="M_solar [M_sun]",
    title="Numerical / analytical median SNR ratio",
    colorbar_title="log10 ratio", xscale=:log10,
)


In [ ]:
median_snr_difference_summary = print_difference_summary(
    "median network SNR grid numerical vs analytical",
    median_snr_matrix_num,
    median_snr_matrix,
)
